# Housing 


## Importing the libraries needed. 
- **Pandas** for loading the data, filtering, sorting, grouping and cleaning the data tables. 
- **Numpy** for mathematical operations.
- **Matplotlib** for visualization. 
- **Sklearn** for splitting the data
- **XGboost** for the regression model

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import time

## Reading the data
Read the *train.csv* and *test.csv* file.

In [15]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)

train.head()

train shape: (1460, 81)
test shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Exploring the dataset
Looking at the number of rows and columns and their data types

In [16]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

## Combine train and test for cleaning
Separate the target and the id column. The remaining columns stack into a DataFrame

In [17]:
y = train['SalePrice'].reset_index(drop=True)

train_ids = train['Id']
test_ids = test['Id']

train_features = train.drop(columns=['Id', 'SalePrice'])
test_features = test.drop(columns=['Id'])

all_data = pd.concat([train_features, test_features], axis=0, ignore_index=True)

print('all_data shape:', all_data.shape)

all_data shape: (2919, 79)


## Check for missing values
Looks for columns with missing values. Check which columns are missing on purpose and which one's are supposed to have values. 

In [18]:
missing = all_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(all_data) * 100).round(1)

missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary

,missing_count,missing_pct
PoolQC,2909,99.7
MiscFeature,2814,96.4
Alley,2721,93.2
Fence,2348,80.4
MasVnrType,1766,60.5
FireplaceQu,1420,48.6
LotFrontage,486,16.6
GarageQual,159,5.4
GarageYrBlt,159,5.4
GarageCond,159,5.4


Some of the columns have NaN meaning the feature does not exist. We fill the categorical one's with **None** and the numeric ones with **0**.

In [19]:
#Columns where NaN = "does not have this feature"
none_cols_categorical = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

for col in none_cols_categorical:
    all_data[col] = all_data[col].fillna('None')

# Numeric columns where NaN = "does not have this feature"
none_cols_numeric = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]

for col in none_cols_numeric:
    all_data[col] = all_data[col].fillna(0)

print('Remaining missing values after handling "feature does not exist" columns:')
print(all_data.isnull().sum().sum())

Remaining missing values after handling "feature does not exist" columns:
499


Breakdown of the remaining values that are **missing** after the ones that don't have the feature have been handled

In [20]:
remaining_missing = all_data.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
remaining_missing

LotFrontage    486
MSZoning         4
Utilities        2
Functional       2
Exterior1st      1
Exterior2nd      1
Electrical       1
KitchenQual      1
SaleType         1
dtype: int64

In [21]:
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)

all_data['MSSubClass'].unique()[:10]

array(['60', '20', '70', '50', '190', '45', '90', '120', '30', '85'],
      dtype=object)

In [22]:
all_data_pre_ordinal = all_data.copy()
n_train = train.shape[0]


## Ordinal encoding 

In [23]:
%%time

quality_map = {
    'None': 0,
    'Po': 1,
    'Fa': 2,
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

quality_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

for col in quality_cols:
    all_data[col] = all_data[col].map(quality_map)

all_data[quality_cols].head()


CPU times: user 9.78 ms, sys: 78 μs, total: 9.86 ms
Wall time: 9.06 ms


,ExterQual,ExterCond,BsmtQual,BsmtCond,HeatingQC,KitchenQual,FireplaceQu,GarageQual,GarageCond,PoolQC
0,4,3,4,3,5,4.0,0,3,3,0
1,3,3,4,3,5,3.0,3,3,3,0
2,4,3,4,3,5,4.0,3,3,3,0
3,3,3,3,4,4,4.0,4,3,3,0
4,4,3,4,3,5,4.0,3,3,3,0


In [24]:
%%time

bsmt_exposure_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
all_data['BsmtExposure'] = all_data['BsmtExposure'].map(bsmt_exposure_map)

bsmt_fin_type_map = {
    'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6
}
all_data['BsmtFinType1'] = all_data['BsmtFinType1'].map(bsmt_fin_type_map)
all_data['BsmtFinType2'] = all_data['BsmtFinType2'].map(bsmt_fin_type_map)

garage_finish_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
all_data['GarageFinish'] = all_data['GarageFinish'].map(garage_finish_map)

all_data[['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish']].head()

CPU times: user 5.58 ms, sys: 970 μs, total: 6.55 ms
Wall time: 5.74 ms


,BsmtExposure,BsmtFinType1,BsmtFinType2,GarageFinish
0,1,6,1,2
1,4,5,1,2
2,2,6,1,2
3,1,5,1,1
4,3,6,1,2


## One-hot encoding

In [25]:
remaining_object_cols = all_data.select_dtypes(include='object').columns.tolist()
print(f'{len(remaining_object_cols)} nominal categorical columns left to one-hot encode:')
print(remaining_object_cols)

30 nominal categorical columns left to one-hot encode:
['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [26]:
%%time

all_data = pd.get_dummies(all_data, columns=remaining_object_cols, drop_first=True)

print('all_data shape after one-hot encoding:', all_data.shape)
all_data.head()

all_data shape after one-hot encoding: (2919, 228)
CPU times: user 25.3 ms, sys: 1.93 ms, total: 27.2 ms
Wall time: 26.1 ms


,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,65.0,8450,7,5,2003,2003,196.0,4,3,4,...,False,False,False,False,True,False,False,False,True,False
1,80.0,9600,6,8,1976,1976,0.0,3,3,4,...,False,False,False,False,True,False,False,False,True,False
2,68.0,11250,7,5,2001,2002,162.0,4,3,4,...,False,False,False,False,True,False,False,False,True,False
3,60.0,9550,7,5,1915,1970,0.0,3,3,3,...,False,False,False,False,True,False,False,False,False,False
4,84.0,14260,8,5,2000,2000,350.0,4,3,4,...,False,False,False,False,True,False,False,False,True,False


## Train, test split

In [27]:
X = all_data.iloc[:n_train, :].reset_index(drop=True)
X_test_kaggle = all_data.iloc[n_train:, :].reset_index(drop=True)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_test_kaggle shape:', X_test_kaggle.shape)

X shape: (1460, 228)
y shape: (1460,)
X_test_kaggle shape: (1459, 228)


## Train, validation split

In [28]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)

X_train shape: (1168, 228)
X_val shape: (292, 228)


## Train a XGBRegressor model

In [29]:
%%time

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

CPU times: user 16.9 s, sys: 43.3 ms, total: 17 s
Wall time: 1.17 s


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
